In [ ]:
import pyelsed
import cv2
import numpy as np
import glob
from scipy.spatial.transform import Rotation as R
import os
import matplotlib.pyplot as plt
import json
import open3d as o3d
from skimage.measure import LineModelND, ransac
from sklearn.cluster import KMeans
# from collections import Counter
from joblib import Parallel, delayed
from scipy import stats
params ={
    "perturb_len":6,
}

In [ ]:
scene_list = ["a1d9da703c","689fec23d7","c173f62b15","69e5939669"]
scene_id = scene_list[2]
data_query = np.load(f'/data1/home/lucky/IROS25/SCORE/line_map_extractor/out/{scene_id}/query/{scene_id}_query_data.npy',allow_pickle=True).item()
scene_line_2D_params = data_query['scene_line_2D_params']
scene_line_2D_points = data_query['scene_line_2D_points']
scene_line_2D_semantic_ids = data_query['scene_line_2D_semantic_ids']
scene_line_2D_end_points = data_query['scene_line_2D_end_points']
id_label_dict = data_query['id_label_dict']
id_label_dict[0] = 'unlabeled'
mask_folder = f'/data1/home/lucky/IROS25/semantic_prediction/processed_dataset_689_mask/'
mask_files = glob.glob(mask_folder + '*.npy')
mask_files.sort()
###
scene_line_2D_semantic_ids_new={}
scene_missing_ratio={}
scene_wrong_ratio={}
wrong_labels = {}
for mask_file in mask_files:
    image_sem_mask = np.load(mask_file)
    base_name=os.path.basename(mask_file).split(".")[0]
    if base_name not in scene_line_2D_points:
        continue
    line_2D_semantic_id = scene_line_2D_semantic_ids[base_name]
    line_2D_semantic_ids_new=[]
    missing_count = 0
    wrong_count = 0
    print("processing",base_name)
    line_2D_points = scene_line_2D_points[base_name]
    num_lines = len(line_2D_points)
    for j in range(num_lines):
        x = line_2D_points[j][0]
        y = line_2D_points[j][1]
        v = np.array([x[0]-x[-1],y[0]-y[-1]])
        v = v/np.linalg.norm(v)
        perturb_list = np.array([0,-1,1])*params['perturb_len']
        old_label = line_2D_semantic_id[j]
        semantic_buffer=[]
        for perturb_l in perturb_list:
            x_p = x + int(perturb_l*v[0])
            y_p = y + int(perturb_l*v[1])
            idx_valid0 = np.logical_and(x_p>=0,x_p<image_sem_mask.shape[1])
            idx_valid1 = np.logical_and(y_p>=0,y_p<image_sem_mask.shape[0])
            idx_valid = np.logical_and(idx_valid0,idx_valid1)
            x_p = x_p[idx_valid].astype(int)
            y_p = y_p[idx_valid].astype(int)
            for i in range(len(x_p)):
                if image_sem_mask[y_p[i],x_p[i]]==0:
                    continue
                semantic_buffer.append(image_sem_mask[y_p[i],x_p[i]])     
        if semantic_buffer==[]:  # no semantic label assigned
            new_label=0
            missing_count+=1
        else:
            new_labels = np.unique(semantic_buffer)
            if old_label not in new_labels: # wrong label assigned
                wrong_count+=1
                new_label = new_labels[0] # take the first label as the new label
            else:
                new_label = old_label
        #
        line_2D_semantic_ids_new.append(new_label)
        if new_label != old_label: # record the wrong prediction
            if (id_label_dict[old_label] + ' ' + id_label_dict[new_label]) not in wrong_labels:
                wrong_labels[id_label_dict[old_label] + ' ' + id_label_dict[new_label]] = 1
            else:
                wrong_labels[id_label_dict[old_label] + ' ' + id_label_dict[new_label]] += 1
            print(f'{base_name},line_{str(j)}, old label:{id_label_dict[old_label]}, new label:{id_label_dict[new_label]}')
    # statistics for this image          
    scene_line_2D_semantic_ids_new[base_name]=line_2D_semantic_ids_new
    scene_missing_ratio[base_name]=missing_count/num_lines
    scene_wrong_ratio[base_name]=wrong_count/num_lines
    print(f'{base_name}, missing ratio: {missing_count/num_lines}, wrong ratio: {wrong_count/num_lines}')

KeyboardInterrupt: 

In [ ]:
sorted_wrong_labels = []
for key in wrong_labels:
    sorted_wrong_labels.append((key,wrong_labels[key]))
sorted_wrong_labels.sort(key=lambda x:x[1],reverse=True)
for wrong_label in sorted_wrong_labels:
    print(wrong_label)

('chair unlabeled', 138)
('chair table', 129)
('chair box', 79)
('wall unlabeled', 78)
('wall door', 33)
('chair floor', 23)
('table unlabeled', 23)
('whiteboard wall', 21)
('wall ceiling', 20)
('banner unlabeled', 19)
('heater unlabeled', 19)
('floor unlabeled', 18)
('cabinet unlabeled', 17)
('ceiling unlabeled', 17)
('window door', 16)
('chair mat', 16)
('floor table', 15)
('banner wall', 15)
('shelf cabinet', 12)
('shelf unlabeled', 10)
('table board', 9)
('windowsill unlabeled', 8)
('roller blinds shelf', 8)
('banner poster', 7)
('wall window', 7)
('table mat', 7)
('roller blinds unlabeled', 7)
('box unlabeled', 7)
('floor door', 6)
('chair wall', 6)
('cabinet wall', 6)
('ceiling door', 6)
('wall chair', 6)
('roller blinds wall', 6)
('door window', 5)
('whiteboard unlabeled', 5)
('floor mat', 5)
('windowsill window', 5)
('ceiling ceiling lamp', 5)
('wall shelf', 4)
('table floor', 4)
('window unlabeled', 4)
('box container', 4)
('cabinet whiteboard', 4)
('roller blinds whiteboard',

In [ ]:
data_query_pred = data_query
data_query_pred['scene_line_2D_semantic_ids'] = scene_line_2D_semantic_ids_new
data_query_pred['scene_label_missing_ratio'] = scene_missing_ratio
data_query_pred['scene_wrong_ratio'] = scene_wrong_ratio
np.save(f'/data1/home/lucky/IROS25/SCORE/line_map_extractor/out/{scene_id}/query/{scene_id}_query_data_pred.npy',data_query_pred)